In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tbparse

In [ ]:
final_csv_path = './result_csvs_and_pdfs/FINAL.csv'
final_df = pd.read_csv(final_csv_path)
final_df

In [ ]:
final_df['label'] = final_df[['model','train_data','test_data']].apply(lambda x:'\n'.join(x), axis=1)
final_df['percent_epochs'] = final_df['train_epochs']/final_df['max_epochs']
final_df.columns

In [ ]:
# get dataframe from lightning log
def get_df_from_log(log_path):
    reader = tbparse.SummaryReader(log_path)
    return reader.scalars

In [ ]:
# get dfs from paths
hyperparam_dfs = {}
train_dfs = {}
test_dfs = {}

for index,row in final_df.iterrows():
    h_df = get_df_from_log(row.hyperparam_log_path)
    hyperparam_dfs[(row.model,row.train_data,row.test_data)] = h_df
    train_df = get_df_from_log(row.train_log_path)
    train_dfs[(row.model,row.train_data,row.test_data)] = train_df
    test_df = get_df_from_log(row.test_log_path)
    test_dfs[(row.model,row.train_data,row.test_data)] = test_df

In [ ]:
# bar plots
x_measures = {
  "Training Runtime (in secs)": "train_walltime",
  "Test AUROC": "test_auroc",
  "Test Loss": "test_loss",
  "Learning Rate": "learning_rate",
  "Batch Size": "batch_size",
  "Gradient Accumulation": "accum_grad_batches",
  "Max Epochs": "max_epochs",
  "Train Epochs": "train_epochs",
  "Train Steps": "train_steps",
  "Percent of Max Epochs Used": "percent_epochs",
}

for x_title,x_col in x_measures.items():
    sns.barplot(x="label", y=f"{x_col}", data=final_df, hue="label", palette='coolwarm')
    plt.xlabel("(Model,Training Data,Test Data)")
    plt.ylabel(f"{x_title}")
    plt.title(f"{x_title} by Model")
    plt.xticks(rotation=45)
    plt.savefig(f'result_csvs_and_pdfs/{x_col}.barplot.pdf')
    plt.show()

In [ ]:
# line plots over epochs (using lightning logs)

x_measures = {
  "Training Loss Across Epochs": "train_loss_epoch",
  "Training Loss Across Steps": "train_loss_step",
  "Positive Prediction Average": "pos_prediction_avg",
  "Negative Prediction Average": "neg_prediction_avg",
}

for x_title,x_col in x_measures.items():
    for key,df in train_dfs.items():
        df_tag = df[df.tag == x_col]
        plt.plot(df_tag.step, df_tag.value, label=key)

    plt.xlabel("Epochs")
    plt.ylabel(f"{x_title}")
    plt.title(f"{x_title} by Model")
    plt.xticks(rotation=45)

    plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=2)
    # plt.tight_layout()
    plt.grid()
    plt.savefig(f'result_csvs_and_pdfs/{x_col}.lineplot.pdf')
    plt.show()